# Exercises: Bayesian Networks from Knowledge Graphs

These exercises accompany the notebook *Bayesian Networks from Knowledge Graphs*. They use the same HALD graph and the same network, with variables Gene $G$, Neighbour gene $N$, Disease $Z$ and Metabolite $L$ and the factorisation

$$P(G,N,Z,L)=P(G)\,P(N\vert G)\,P(Z\vert G,N)\,P(L\vert Z).$$

Each exercise says what you are asked to do, and has a part to do **by hand** and a part to do **in code**. In the code cells, the lines that you must write are marked `...  # TODO`. The exercises take about an hour in total.

| Exercise | Skill | Time |
|:--|:--|:--|
| 1 | Estimating CPTs from a graph by hand, on a toy graph | 15 min |
| 2 | Choosing state spaces; leakage from the withheld set | 10 min |
| 3 | Choosing smoothing constants *without* peeking at the test set | 15 min |
| 4 | Conditional independence by marginalising the joint distribution | 10 min |
| 5 | Explaining a recommendation with SPARQL | 10 min |

## Setup

The three code cells below are copied, unchanged, from the main notebook. They load HALD with Owlready2, extract the links with SPARQL, build the count matrices, split off the withheld links, build the network (`model`), and define the inference function `posterior_genes`, the function `recommend` and the evaluation function `rank_metrics`. Run them once, in order. (They take about ten seconds.) The file `HALD.rdf` must be in the same folder as this notebook.

In [ ]:
import collections
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
from owlready2 import World

HALD_FILE = "HALD.rdf"          # the RDF/XML file, in the same folder as this notebook
SEED = 2024                     # seed for the train/withheld split
METAB = ["Lipid", "Carbohydrate"]   # the node classes that we treat as "metabolites"

world = World()                                                   # a fresh quadstore, so that re-running this cell starts clean
onto = world.get_ontology(Path(HALD_FILE).resolve().as_uri()).load()
print(f"{len(list(onto.individuals())):,} individuals, {len(list(onto.object_properties())):,} object properties, "
      f"{len(list(onto.classes()))} classes")

PREFIXES = "PREFIX h: <http://www.dummy.info/new.owl#>\n"

def sparql(query):
    """Run a SPARQL query against the loaded graph; returns a list of rows."""
    return list(world.sparql(PREFIXES + query))

In [ ]:
def link_table(types_a, types_b):
    """Distinct pairs (a, b), a of a class in types_a and b of a class in types_b, linked by ANY relation in
    EITHER direction. The UNION makes the query symmetric, which collapses the relation vocabulary to a single
    'linked' relation and ignores direction."""
    values = lambda var, types: f"VALUES {var} {{ " + " ".join("h:" + t for t in types) + " }"
    query = f"""SELECT DISTINCT ?a ?b WHERE {{
        {values('?type_a', types_a)} {values('?type_b', types_b)}
        ?a a ?type_a . ?b a ?type_b .
        {{ ?a ?rel ?b }} UNION {{ ?b ?rel ?a }}
        ?rel a owl:ObjectProperty .
        FILTER (?a != ?b) }}"""
    out = pd.DataFrame([(a.name, b.name) for a, b in sparql(query)], columns=["source", "target"])
    return out.sort_values(["source", "target"]).reset_index(drop=True)

gene_disease = link_table(["Gene"], ["Disease"])
gene_gene = link_table(["Gene"], ["Gene"])                      # symmetric, so both directions are present
disease_metabolite = link_table(["Disease"], METAB)
gene_metabolite = link_table(["Gene"], METAB)                   # NOT used by the model: kept for a later check

print(f"gene-disease: {len(gene_disease):,}   gene-gene (both directions): {len(gene_gene):,}   disease-metabolite: {len(disease_metabolite):,}")

# Universe: genes with at least one disease link; diseases linked to at least two genes
# (so that a disease can lose one edge to the withheld set and still have evidence left).
n_links = gene_disease.groupby("target").size()          # genes linked to each disease
diseases = sorted(n_links[n_links >= 2].index)
gene_disease = gene_disease[gene_disease.target.isin(diseases)]
genes = sorted(gene_disease.source.unique())
g_idx = {g: i for i, g in enumerate(genes)}
z_idx = {z: i for i, z in enumerate(diseases)}
nG, nZ = len(genes), len(diseases)

# Withhold a random 20% of the gene-disease links. Sort first, so the split depends only on SEED.
pairs = np.array([(g_idx[g], z_idx[z]) for g, z in zip(gene_disease.source, gene_disease.target)])
pairs = pairs[np.lexsort((pairs[:, 1], pairs[:, 0]))]
perm = np.random.default_rng(SEED).permutation(len(pairs))
n_held = int(0.2 * len(pairs))
held, train = pairs[perm[:n_held]], pairs[perm[n_held:]]
print(f"{nG:,} genes, {nZ:,} diseases;  {len(train):,} training links, {len(held):,} withheld links")

# Count matrices (sparse). Rows are indexed by the first letter(s) of the name, as in the rest of the notes.
CZG = sp.csr_matrix((np.ones(len(train)), (train[:, 1], train[:, 0])), shape=(nZ, nG))   # CZG[z, g] = 1 if training link g-z

gg = [(g_idx[a], g_idx[b]) for a, b in zip(gene_gene.source, gene_gene.target) if a in g_idx and b in g_idx]
NB = sp.csr_matrix((np.ones(len(gg)), tuple(zip(*gg))), shape=(nG, nG))                  # NB[n, g] = 1 if genes n and g are linked
NB.data[:] = 1

# Metabolites (lipids and carbohydrates): keep the K most-linked metabolites (among universe diseases) and merge the rest into 'other'.
dm = disease_metabolite[disease_metabolite.source.isin(diseases)]
K = 20
top_met = dm.target.value_counts().head(K).index.tolist()
metabolites = top_met + ["other"]
l_idx = {l: i for i, l in enumerate(metabolites)}
CLZ = sp.lil_matrix((len(metabolites), nZ))                                                # CLZ[l, z] = 1 if disease z has metabolite m
for z, l in zip(dm.source, dm.target):
    CLZ[l_idx.get(l, K), z_idx[z]] = 1
CLZ = CLZ.tocsr()
nL = len(metabolites)

summary = pd.DataFrame({
    "relation": ["gene-disease (training)", "gene-gene", "disease-metabolite (kept)"],
    "number of links": [CZG.nnz, NB.nnz // 2, CLZ.nnz],
    "used for": ["P(G), P(Z|G,N)", "P(N|G), P(Z|G,N)", "P(L|Z)"]})
print(f"{K} named metabolites cover {dm.target.isin(top_met).sum()} of {len(dm)} disease-metabolite links")

In [ ]:
def build_model(CZG, NB, CLZ, lam=0.5, b=1.0, c=1.0):
    """Estimate the four CPTs from count matrices. Nothing larger than (diseases x genes) is ever stored.

    CZG[z, g]  training links between disease z and gene g      NB[n, g]  links between genes n and g
    CLZ[l, z]  links between metabolite l and disease z
    lam        weight given to the neighbour genes when a gene already has training links of its own
    b, c       add-b / add-c (Laplace-style) smoothing for P(Z|N) and P(L|Z)
    """
    nZ, nG = CZG.shape
    nL = CLZ.shape[0]
    d = np.asarray(CZG.sum(axis=0)).ravel()                      # training links per gene
    nz = np.asarray(CZG.sum(axis=1)).ravel()                     # training links per disease
    PZ0 = (nz + 1) / (nz.sum() + nZ)                             # P(Z): smoothed marginal

    PG = (d + 1) / (d.sum() + nG)                                # P(G): add-one smoothed share of links

    deg = np.asarray(NB.sum(axis=0)).ravel()                     # P(N|G=g): uniform over the neighbours of g ...
    PN_G = NB @ sp.diags(1 / np.maximum(deg, 1))                 # (sparse, columns sum to 1)
    PN = np.asarray(NB.sum(axis=1)).ravel() + 0.0                # ... and for a gene with no neighbours, the marginal
    PN = PN / PN.sum() if PN.sum() > 0 else np.full(nG, 1 / nG)  #     distribution of neighbours

    # P(Z|N): what a gene n says about diseases, from its own training links, smoothed towards P(Z)
    PZ_N = (CZG.toarray() + b * PZ0[:, None]) / (d[None, :] + b)

    # Mediator part of P(Z|G) = sum_n P(n|g) P(Z|n): a sparse-times-dense product
    via_N = np.asarray((PZ_N @ PN_G))                            # (nZ x nG)
    iso = deg == 0
    via_N[:, iso] = (PZ_N @ PN)[:, None]                         # genes with no neighbours use the marginal

    lam_g = np.where(d > 0, lam, 1.0)                            # a gene with no training links relies on N entirely
    own = CZG.toarray() / np.maximum(d, 1)[None, :]              # empirical P(Z|g) from the gene's own links
    PZ_G = (1 - lam_g) * own + lam_g * via_N                     # P(Z|G): P(Z|G,N) marginalised over N (see text)

    PL_Z = (CLZ.toarray() + c * ((np.asarray(CLZ.sum(axis=1)).ravel() + 1) / (CLZ.sum() + nL))[:, None])
    PL_Z = PL_Z / PL_Z.sum(axis=0, keepdims=True)                # P(L|Z): smoothed towards the marginal P(L)
    return dict(PG=PG, PN_G=PN_G, PN=PN, PZ_N=PZ_N, PZ_G=PZ_G, PL_Z=PL_Z, PZ0=PZ0, d=d, lam_g=lam_g, own=own, deg=deg)

model = build_model(CZG, NB, CLZ)
print("P(G) sums to", model["PG"].sum().round(6),
      "| every column of P(Z|G) sums to 1:", np.allclose(model["PZ_G"].sum(axis=0), 1),
      "| every column of P(L|Z) sums to 1:", np.allclose(model["PL_Z"].sum(axis=0), 1))

def likelihood(model, z):
    """P(Z=z | G=g) for every gene g at once: one row of the (diseases x genes) matrix P(Z|G)."""
    return model["PZ_G"][z]

def posterior_genes(model, z, uniform_prior=False):
    """P(G | Z=z) by Bayes' rule: proportional to P(G) P(z|G). Exact, and uses only the stored pieces."""
    prior = np.full(len(model["PG"]), 1 / len(model["PG"])) if uniform_prior else model["PG"]
    post = prior * likelihood(model, z)
    return post / post.sum()

def diseases_for_gene(model, g):
    """P(Z | G=g): a column of P(Z|G), with no inference needed."""
    return model["PZ_G"][:, g]

# Training-link lookup, used to exclude links that are already known
known = CZG.toarray() > 0
held_by_disease = collections.defaultdict(list)
for g, z in held:
    held_by_disease[z].append(g)

def recommend(z, k=5):
    p = posterior_genes(model, z)
    p = np.where(known[z], -np.inf, p)            # exclude genes already linked to z in the training graph
    top = np.argsort(-p)[:k]
    return pd.DataFrame({"gene": [genes[g] for g in top], "P(G|Z=z)": p[top],
                         "withheld link?": [g in held_by_disease[z] for g in top]})



def rank_metrics(scores, pairs, by="gene", known_mat=None):
    """Rank the withheld link for each (g, z) pair. scores[z, g] is a score for every candidate.
    by='gene': candidates are genes, for the given disease (query: P(G|Z=z)); by='disease': candidates are diseases
    for the given gene. Candidates that already have a TRAINING link to the query are removed (filtered ranking)."""
    kn = known if known_mat is None else known_mat              # which links count as already known
    ranks = []
    for g, z in pairs:
        s = scores[z].copy() if by == "gene" else scores[:, g].copy()
        mask = kn[z].copy() if by == "gene" else kn[:, g].copy()
        target = g if by == "gene" else z
        mask[target] = False
        s[mask] = -np.inf
        ranks.append(1 + np.sum(s > s[target]) + 0.5 * (np.sum(s == s[target]) - 1))   # ties share the rank
    r = np.array(ranks)
    return {"MRR": np.mean(1 / r), "Hits@10": np.mean(r <= 10), "Hits@100": np.mean(r <= 100), "median rank": np.median(r)}

## 1. A toy knowledge graph, by hand

The calculation in the main notebook is the same as the one that you can do on a very small graph with pencil and paper. Doing it once by hand is the best way to understand what the code is doing. Here is a toy graph with four genes ($A$, $B$, $C$, $D$), three diseases ($x$, $y$, $z$) and two metabolites.

* Training gene–disease links: $A$–$x$, $A$–$y$, $B$–$x$, $C$–$z$. Gene $D$ has no training link.
* Gene–gene links (undirected): $A$–$B$, $B$–$C$, $C$–$D$.
* Disease–metabolite links: $x$–$m_1$, $y$–$m_2$, $z$–$m_1$. (The metabolite is not needed for this exercise.)

For reference, these are the estimators from the main notebook. $d_g$ is the number of training links of gene $g$, $D$ is the total number of training links, $n_z$ is the number of training links of disease $z$, and $c(g,z)$ is 1 if there is a training link between $g$ and $z$ and 0 otherwise.

$$P(G=g)=\frac{d_g+1}{D+\vert G\vert}\qquad P(Z=z)=\frac{n_z+1}{D+\vert Z\vert}\qquad P(N=n\vert G=g)=\frac{1}{\text{number of neighbours of }g}\ \text{(for each neighbour)}$$

$$P(Z=z\vert N=n)=\frac{c(n,z)+b\,P(Z=z)}{d_n+b}\qquad P(Z=z\vert G=g)=(1-\lambda_g)\frac{c(g,z)}{d_g}+\lambda_g\sum_nP(n\vert g)P(z\vert n)$$

with $\lambda_g=\lambda$ if $d_g>0$ and $\lambda_g=1$ if $d_g=0$. In this exercise $\lambda=0.5$ and $b=1$. The posterior is $P(G\vert Z=z)\propto P(G)P(Z=z\vert G)$.

**Work through these by hand.** Write your answers as fractions where you can.

a) Compute $P(G)$, the prior over $A,B,C,D$. (There are four training links, and the number of links per gene is 2, 1, 1, 0.)

b) Compute the smoothed marginal $P(Z)$ and then $P(Z\vert N=B)$, a vector over $x,y,z$. Gene $B$ has one training link, to $x$.

c) Compute $P(Z=x\vert G=B)$. Gene $B$ has neighbours $A$ and $C$, each with probability $\tfrac12$, so you will need $P(Z=x\vert N=A)$ and $P(Z=x\vert N=C)$ as well.

d) The code will print $P(Z=x\vert G=g)$ for all four genes. Use these, with your $P(G)$, to compute the posterior $P(G\vert Z=x)$ by Bayes' rule.

e) Which gene would the network recommend for disease $x$ if we exclude the genes that already have a training link to $x$?

Enter your hand-calculated answers in the cells below, and the code will check them against the program of the main notebook.

In [ ]:
# A toy knowledge graph: 4 genes, 3 diseases, 2 metabolites.
toy_genes, toy_diseases, toy_metabolites = ["A", "B", "C", "D"], ["x", "y", "z"], ["m1", "m2"]
gene_disease_links = [("A", "x"), ("A", "y"), ("B", "x"), ("C", "z")]          # training gene-disease links
gene_gene_links = [("A", "B"), ("B", "C"), ("C", "D")]                         # undirected gene-gene links
disease_metabolite_links = [("x", "m1"), ("y", "m2"), ("z", "m1")]

ig = {g: i for i, g in enumerate(toy_genes)}
iz = {z: i for i, z in enumerate(toy_diseases)}
il = {l: i for i, l in enumerate(toy_metabolites)}
CZG_t = sp.csr_matrix(([1.0] * 4, ([iz[z] for g, z in gene_disease_links], [ig[g] for g, z in gene_disease_links])), shape=(3, 4))
NB_t = sp.csr_matrix(([1.0] * 6, ([ig[a] for a, b in gene_gene_links] + [ig[b] for a, b in gene_gene_links],
                                  [ig[b] for a, b in gene_gene_links] + [ig[a] for a, b in gene_gene_links])), shape=(4, 4))
CLZ_t = sp.csr_matrix(([1.0] * 3, ([il[l] for z, l in disease_metabolite_links], [iz[z] for z, l in disease_metabolite_links])), shape=(2, 3))

# lam = 0.5 and b = 1 (the smoothing constants of the notes)
toy1 = build_model(CZG_t, NB_t, CLZ_t, lam=0.5, b=1.0, c=1.0)

In [ ]:
# a) P(G), in the order A, B, C, D
PG_hand = np.array([..., ..., ..., ...])  # TODO enter your values
print("P(G) correct:", np.allclose(PG_hand, toy1["PG"]))

In [ ]:
# b) P(Z | N=B), in the order x, y, z
PZ_B_hand = np.array([..., ..., ...])  # TODO enter your values
print("P(Z|N=B) correct:", np.allclose(PZ_B_hand, toy1["PZ_N"][:, ig["B"]]))

In [ ]:
# c) P(Z=x | G=B)
PxB_hand = ...  # TODO enter your value
print("P(Z=x|G=B) correct:", np.isclose(PxB_hand, toy1["PZ_G"][iz["x"], ig["B"]]))

In [ ]:
# d) The program's values of P(Z=x | G=g) for g = A, B, C, D, and your posterior P(G | Z=x)
print("P(Z=x|G=g):", toy1["PZ_G"][iz["x"]].round(4))
post_hand = np.array([..., ..., ..., ...])  # TODO enter your posterior values (they should sum to 1)
print("P(G|Z=x) correct:", np.allclose(post_hand, posterior_genes(toy1, iz["x"])))

In [ ]:
# e) Exclude the genes with a training link to x, then take the most probable gene that is left
known_t = CZG_t.toarray() > 0
candidates = ...  # TODO the posterior with the already-linked genes set to -inf
print("Recommended gene:", toy_genes[int(np.argmax(candidates))])

**Questions.**

1. Gene $D$ has no training links and no evidence of its own, yet $P(G=D)>0$ and $P(Z\vert G=D)$ is a proper distribution. Where does each of these come from?
2. Gene $A$ has the largest posterior for $x$, yet it is excluded. Why do we exclude genes that are already linked, and what would happen to an evaluation if we did not?
3. Look at the genes that remain. What kind of evidence supports the recommended gene, and what supports the other one? Which of the two do you trust more, and why?

## 2. State spaces and leakage

**Reducing a state space.** The metabolite variable $L$ has one state for each of the $K$ most linked metabolites, plus "other". The choice of $K$ is a trade-off: a larger $K$ keeps more information, but each state is then estimated from fewer links. The setup cells have already built `dm`, the table of (disease, metabolite) links in our universe. Use it to study the trade-off.

a) Count the links per metabolite (`link_counts` is given), and compute `coverage`: the share of *all* disease–metabolite links that fall on the top $K$ metabolites, for every $K$. Find the smallest $K$ that covers half of the links, and the smallest that covers three quarters.

b) The table then lists, for four values of $K$, the number of cells in $P(L\vert Z)$ and the number of links per cell. What do you conclude?

**Leakage and cold start.** We withheld a random 20% of the gene–disease links, so a disease with only two links has a small chance of losing *both*.

c) By hand: if each link is withheld independently with probability 0.2, what is the probability that a disease with exactly two links in the universe loses both? What is it for a disease with five links?

d) The expected number of diseases left with *no* training link is $\sum_z 0.2^{k_z}$, where $k_z$ is the number of links of disease $z$ in the universe. Compute it, and compare it with the actual number.

In [ ]:
link_counts = dm.target.value_counts()                      # links per metabolite, most linked first

coverage = ...  # TODO cumulative share of all links covered by the top K metabolites (a numpy array indexed by K-1)

K_for_half = ...  # TODO smallest K that covers at least 50% of the links
K_for_three_quarters = ...  # TODO smallest K that covers at least 75% of the links
print(f"{len(link_counts)} metabolites in all; the top {K_for_half} cover 50% of the links and the top {K_for_three_quarters} cover 75%")

plt.figure(figsize=(5, 3))
plt.plot(np.arange(1, 201), coverage[:200])
plt.xlabel("K, the number of named metabolites"); plt.ylabel("share of links covered")
plt.show()

rows = []
for K_try in (5, 20, 50, 100):
    cells = nZ * (K_try + 1)                                 # size of P(L|Z): K_try named metabolites plus 'other'
    covered = coverage[K_try - 1] * link_counts.sum()        # links that fall on a named metabolite
    rows.append((K_try, cells, int(covered), covered / cells))
display(pd.DataFrame(rows, columns=["K", "cells in P(L|Z)", "links on named metabolites", "links per cell"]).round(3))

In [ ]:
links_per_disease = np.bincount(pairs[:, 1], minlength=nZ)   # links per disease in the whole universe, before withholding
expected_empty = ...  # TODO expected number of diseases with every link withheld
actual_empty = ...  # TODO number of diseases whose row of the training count matrix CZG is all zeros
print(f"expected diseases with no training link: {expected_empty:.1f}   actual: {actual_empty}")

**Questions.**

1. Choose a value of $K$ for the metabolites and justify it using your results.
2. What does the "links per cell" column tell you about the reliability of the estimated $P(L\vert Z)$?
3. A disease that ends up with no training link is still in the network. What does the network do for it, and could it still be a sensible *query*? (Think about $P(Z)$ and about $P(G\vert Z=z)$ for such a disease.)

## 3. Choosing the smoothing constants without peeking

The network has two smoothing constants: $\lambda$, the weight given to the neighbour genes, and $b$, the strength of the shrinkage of $P(Z\vert n)$ towards $P(Z)$. The main notebook fixed them at $\lambda=0.5$, $b=1$. A better way to choose them is to try several values and keep the best. But *best according to what?*

If you choose the values that give the best scores on the withheld test links, the test is no longer a fair assessment: you have tuned the model to the test set, and the reported performance is optimistic. The correct procedure is to split off a **validation set** from the *training* links, to choose the constants on that, and only then to evaluate once on the withheld links.

The cell below splits the training links into a fitting part and a validation part, and defines `fit_and_evaluate`, which builds the network from the fitting links only and ranks the validation links. (The fitting links count as known, and the withheld test links are never used.)

a) Evaluate a grid of values, $\lambda\in\{0,0.1,0.5,0.9\}$ and $b\in\{0.1,1,10,100\}$, on the validation links, and find the best pair by MRR.

b) Refit with the chosen constants on *all* of the training links and evaluate **once** on the withheld links. Compare with the default values.

In [ ]:
# Split the TRAINING links into a fitting part (80%) and a validation part (20%). The withheld test links are not touched.
rng = np.random.default_rng(SEED + 1)
perm = rng.permutation(len(train))
n_val = int(0.2 * len(train))
val_pairs, fit_pairs = train[perm[:n_val]], train[perm[n_val:]]

def fit_and_evaluate(fit_pairs, eval_pairs, lam, b):
    """Build the network from fit_pairs only, and rank the links in eval_pairs. Links in fit_pairs count as already known."""
    CZ = sp.csr_matrix((np.ones(len(fit_pairs)), (fit_pairs[:, 1], fit_pairs[:, 0])), shape=(nZ, nG))
    m = build_model(CZ, NB, CLZ, lam=lam, b=b)
    return rank_metrics(m["PG"][None, :] * m["PZ_G"], eval_pairs, known_mat=CZ.toarray() > 0)

In [ ]:
rows = {}
for lam in (0.0, 0.1, 0.5, 0.9):
    for b in (0.1, 1.0, 10.0, 100.0):
        rows[(lam, b)] = ...  # TODO evaluate on the validation links, with fit_and_evaluate
sweep = pd.DataFrame(rows).T
sweep.index.names = ["lambda", "b"]
display(sweep.round(3))

best_lam, best_b = ...  # TODO the pair with the best validation MRR
print(f"best on the validation links (by MRR): lambda = {best_lam}, b = {best_b}")

In [ ]:
# Refit on ALL the training links and evaluate once on the withheld test links
final = pd.DataFrame({"default (lambda=0.5, b=1)": ...,   # TODO
                      f"chosen (lambda={best_lam}, b={best_b})": ...}).T   # TODO
display(final.round(3))

**Questions.**

1. Why must the validation links be taken from the *training* links, and why is the fitting part smaller than the full training set? What does this do to the absolute numbers in the sweep, compared with the final table?
2. Look at the row $\lambda=0$. Why is it so poor? (Think about what the likelihood is for a disease that is not among a gene's own training links.)
3. The best value of $b$ lies at the edge of the grid. What should you do next?
4. Does the tuned network beat the "prior only" baseline of the main notebook (MRR 0.066, Hits@100 0.382)? Is that a fair comparison?

## 4. Conditional independence by marginalisation

The network's structure *asserts* conditional independences. For a small universe we can build the whole joint distribution as a four-dimensional array and *check* them by summing, as we did in the main notebook. The cell below builds a sub-universe of 12 genes, 8 diseases and 4 metabolites, chosen so that it contains gene–gene links, and the joint $P(G,N,Z,L)$ as an array `J` indexed `[g, n, z, l]`.

Marginalising a variable means summing the array over its axis (`J.sum(axis=...)`); conditioning means dividing by the sum.

a) Compute $P(G\vert Z=z)$ for every $z$: first $P(G,Z)$ by summing over $N$ and $L$, then divide by $P(Z)$.

b) Fix $z=2$ and $l=1$. Compute $P(G\vert Z=z,L=l)$ by summing over $N$ only. Compare with $P(G\vert Z=z)$.

c) Compute $P(G\vert L=l)$, by summing over $N$ and $Z$, and compare with the marginal $P(G)$.

d) Compute $P(G,N)$ and compare with $P(G)P(N)$.

e) Compute $P(N,L\vert Z=z)$ and compare with $P(N\vert Z=z)P(L\vert Z=z)$.

In each case print the largest absolute difference.

In [ ]:
# A small universe: 12 genes, 8 diseases and 4 metabolites, small enough to build the full joint distribution.
rng = np.random.default_rng(0)
usable = np.flatnonzero(model["d"] > 0)                               # genes with training links
hub = usable[np.argmax(model["deg"][usable])]                         # start from the gene with most neighbours,
nbrs = [n for n in NB[:, [hub]].nonzero()[0] if model["d"][n] > 0]    # so that the sub-universe contains gene-gene links
extra = [g for g in rng.permutation(usable) if g != hub and g not in nbrs]
gs = np.sort(np.array(([hub] + nbrs + extra)[:12]))
zs = np.sort(np.argsort(-np.asarray(CZG[:, gs].sum(axis=1)).ravel())[:8])   # the 8 diseases with most links to these genes
ls = np.arange(4)
toy = build_model(CZG[zs][:, gs], NB[gs][:, gs], CLZ[ls][:, zs])

def dense_joint(m):
    """P(G,N,Z,L) = P(G) P(N|G) P(Z|G,N) P(L|Z) as a dense 4-D array, indexed [g, n, z, l]."""
    PN_G = m["PN_G"].toarray()
    PN_G[:, m["deg"] == 0] = m["PN"][:, None]
    PZ_GN = ((1 - m["lam_g"])[None, :, None] * m["own"][:, :, None] + m["lam_g"][None, :, None] * m["PZ_N"][:, None, :])
    return np.einsum("g,ng,zgn,lz->gnzl", m["PG"], PN_G, PZ_GN, m["PL_Z"])

J = dense_joint(toy)
print("shape of the joint [g, n, z, l]:", J.shape, "  sums to", J.sum().round(10))

In [ ]:
# a) P(G | Z=z) for every z: column z of PG_given_Z
PGZ = ...  # TODO sum J over N (axis 1) and L (axis 3)
PG_given_Z = ...  # TODO divide each column by its sum

# b) P(G | Z=z, L=l)
z, l = 2, 1
PGZL = ...  # TODO sum J over N only; the result is indexed [g, z, l]
PG_given_ZL = ...  # TODO
print("max |P(G|Z=z,L=l) - P(G|Z=z)| =", np.abs(PG_given_ZL - PG_given_Z[:, z]).max().round(12))

# c) P(G | L=l)
PG_given_L = ...  # TODO
PG_marginal = ...  # TODO
print("max |P(G|L=l) - P(G)|         =", np.abs(PG_given_L - PG_marginal).max().round(6))

# d) G and N
PGN = ...  # TODO
print("max |P(G,N) - P(G)P(N)|       =", ...)  # TODO

# e) N and L, given Z
PNL_given_Z = ...  # TODO
print("max |P(N,L|Z=z) - P(N|Z=z)P(L|Z=z)| =", ...)  # TODO

**Questions.**

1. Which of the four comparisons gave (essentially) zero, and which did not? Explain each result using the DAG $G\to N$, $G\to Z$, $N\to Z$, $Z\to L$.
2. Part (c) is small here. Does a small number mean that $G$ and $L$ are independent? Why is the result not exactly zero?
3. Suppose that you added an edge $G\to L$ to represent the direct gene–metabolite links in the graph. Which of your results would change? How would the number of free parameters change for the full network of the main notebook?

## 5. Explaining a recommendation with SPARQL

A ranked list does not say *why* a gene is plausible. In the network, the probability $P(Z=z\vert G=g)$ is a sum over the neighbours of $g$, and each term is a *path instance* $g - n - z$, in which the second edge is a training link. The *relation names* on the two edges of each path, the verbs in the original text, are the human-readable evidence, and the graph stores them: we collapsed them in order to count, but we can still look them up.

a) Write a SPARQL query that finds the five relation names that most often link a gene to a disease (in the direction gene $\to$ disease), together with their counts. You will need the `?rel a owl:ObjectProperty` pattern to restrict `?rel` to relations, and `GROUP BY`, `ORDER BY` and `LIMIT`. The prefix `h:` is defined for you by the `sparql` function.

b) The function `relations_between(a, b)` is given below: it returns the relation names that link nodes $a$ and $b$ in either direction. Complete `path_instances(z, g)`, which lists every neighbour $n$ of gene $g$ that has a *training* link to disease $z$, with the weight of the path ($\lambda_g\,P(n\vert g)\,P(z\vert n)$) and the relation names on both edges. Apply it to the top recommendation for the illustrative disease.

In [ ]:
query = """
...  # TODO a SPARQL query for the five most frequent gene -> disease relation names
"""
rows = sparql(query)
display(pd.DataFrame([(r.name, n) for r, n in rows], columns=["relation", "links"]))

In [ ]:
def iri(name):
    """The full IRI of a node, which is safe to use in a query even if the name contains a hyphen or a dot."""
    return "<" + onto.base_iri + name + ">"

def relations_between(a, b):
    """The relation names that link node a to node b, in either direction, from the graph (via SPARQL)."""
    rows = sparql(f"""SELECT ?rel ?direction WHERE {{
        {{ {iri(a)} ?rel {iri(b)} . BIND("->" AS ?direction) }} UNION {{ {iri(b)} ?rel {iri(a)} . BIND("<-" AS ?direction) }}
        ?rel a owl:ObjectProperty }}""")
    return sorted(f"{r.name} ({d})" for r, d in rows)

In [ ]:
# The illustrative disease from the main notebook: few training links, at least one withheld link
cands = [z for z in held_by_disease if known[z].sum() <= 3 and len(held_by_disease[z]) >= 1]
z_demo = sorted(cands, key=lambda z: (-len(held_by_disease[z]), diseases[z]))[0]

def path_instances(z, g):
    """All paths  gene g - neighbour n - disease z  where the n-z link is a TRAINING link, with the relation names."""
    rows = []
    for n in NB[:, [g]].nonzero()[0]:
        if ...:  # TODO n has a training link to z (use the matrix `known`)
            rows.append({"neighbour n": genes[n],
                         "weight": ...,  # TODO lambda_g * P(n|g) * P(z|n), from model["lam_g"], model["PN_G"] and model["PZ_N"]
                         "relations g-n": ", ".join(relations_between(genes[g], genes[n])),
                         "relations n-z": ", ".join(relations_between(genes[n], diseases[z]))})
    return pd.DataFrame(rows).sort_values("weight", ascending=False)

rec = recommend(z_demo)
g_rec = genes.index(rec.gene[0])
print(f"Recommendation for {diseases[z_demo]}: {genes[g_rec]}")
display(path_instances(z_demo, g_rec))

**Questions.**

1. For the top recommendation, which neighbour supplies the evidence? Read the relation names on the two edges. Do they suggest a plausible biological connection, or only that the genes were mentioned together? What does that say about collapsing all relations to "linked"?
2. Choose a *withheld* link for this disease (the main notebook lists them). Does it have a supporting path of this kind in the training graph? What does the answer tell you about whether the network *could* have found it?
3. For a recommendation in a project report, why is the list of paths, with the relation names, more convincing than the probability?